[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madhudream/rag_simple/blob/main/colab/2c_hybrid_search.ipynb)

# 3c — What is hybrid search? BM25 + vectors, fused

Companion notebook to blog post **3c (What is hybrid search?)**. Standalone: rebuilds
post 2a's BM25 and post 2b's vector search over a 9-document corpus, then fuses them
with Reciprocal Rank Fusion (RRF).

In [ ]:
%pip install -q fastembed numpy qdrant-client

## Corpus — everyday sentences + a support-ticket wing

Docs 5–7 are near-twins differing only in an error code: vector search's trap.

In [ ]:
corpus = [
    "The cat sat on the warm windowsill in the sun",                       # doc 0
    "A dog chased the cat around the yard",                                # doc 1
    "Dogs are loyal and love to play fetch in the park",                   # doc 2
    "The park has a pond where ducks swim every morning",                  # doc 3
    "She planted tomatoes and basil in her garden",                        # doc 4
    "Error E-4042 refund transaction declined by the payment gateway",     # doc 5
    "Error E-4043 refund transaction succeeded but receipt email failed",  # doc 6
    "Error E-4044 refund transaction pending manual review",               # doc 7
    "How refunds work a general overview of the refund process",           # doc 8
]

## Expert 1 — the detective (post 2a's BM25, unchanged)

`bm25_search` returns only documents with score > 0 — no word overlap, not retrieved.

In [ ]:
import math
from collections import Counter

def tokenize(text):
    return text.lower().split()

docs = [tokenize(d) for d in corpus]
N = len(docs)
avgdl = sum(len(d) for d in docs) / N

df = Counter()
for d in docs:
    for term in set(d):
        df[term] += 1

def idf(term):
    n = df.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def term_score(term, doc, k1=1.5, b=0.75):
    freqs = Counter(doc)
    if term not in freqs:
        return 0.0
    f = freqs[term]
    dl = len(doc)
    doc_length_norm = 1 - b + b * dl / avgdl
    term_freq_saturation = f * (k1 + 1) / (f + k1 * doc_length_norm)
    return idf(term) * term_freq_saturation

def bm25_search(query):
    scored = [(sum(term_score(t, d) for t in tokenize(query)), i) for i, d in enumerate(docs)]
    scored.sort(reverse=True)
    return [(s, i) for s, i in scored if s > 0]   # zero-score docs are NOT retrieved

## Expert 2 — the map guide (post 2b's vector search, unchanged)

In [ ]:
import numpy as np
from fastembed import TextEmbedding

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
doc_embs = list(model.embed(corpus))

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def vector_search(query):
    q = list(model.embed([query]))[0]
    scored = [(cosine(q, e), i) for i, e in enumerate(doc_embs)]
    scored.sort(reverse=True)
    return scored

## Each expert alone — two opposite failures

- `E-4042`: vector ranks the WRONG twin first (0.468 vs 0.465); BM25 retrieves exactly one doc — the right one.
- `puppy playing outside`: BM25 comes back empty; vector walks straight to the dog docs.

In [ ]:
for q in ["E-4042", "puppy playing outside"]:
    print(f"\n=== Query: {q!r} ===")
    print("BM25:")
    hits = bm25_search(q)
    if not hits:
        print("  (nothing — every score 0.000)")
    for s, i in hits[:4]:
        print(f"  {s:.3f}  doc {i}: {corpus[i]}")
    print("Vector:")
    for s, i in vector_search(q)[:4]:
        print(f"  {s:.3f}  doc {i}: {corpus[i]}")

## Step 1 — Try adding the scores. It fails.

BM25 is unbounded and shifts per query; cosine never leaves [−1, 1]. Adding them,
the louder unit wins — not the righter expert. And there's no stable rescale constant.

In [ ]:
print("BM25 top score for 'error E-4042'   :", round(bm25_search("error E-4042")[0][0], 3))
print("BM25 top score for 'garden tomatoes':", round(bm25_search("garden tomatoes")[0][0], 3))
print("Vector top score (cosine, always)   : <= 1.0")

## Step 2 — Reciprocal Rank Fusion: score by position

```
RRF(doc) = Σ over lists  1 / (60 + rank)
```

Hand-worked for `E-4042` (real lists: BM25 `[5]`, vector `[6, 5, 7, 8, 3]`):

```
        from BM25's list         from vector's list       TOTAL
doc 5   rank 1 → 1/61 = 0.01639  rank 2 → 1/62 = 0.01613  0.03252  ← wins (on BOTH lists)
doc 6   not on it → 0            rank 1 → 1/61 = 0.01639  0.01639
doc 7   not on it → 0            rank 3 → 1/63 = 0.01587  0.01587
```

`points.most_common(top)` = "highest point totals, best-first" — the sorted TOTAL column.

In [ ]:
def rrf_fuse(rankings, k=60, top=4):
    points = Counter()                       # the running-totals column
    for ranking in rankings:                 # one ranked id list per searcher
        for rank, doc_id in enumerate(ranking, start=1):
            points[doc_id] += 1 / (k + rank)
    return points.most_common(top)

def hybrid_search(query, top=4):
    bm25_ids = [i for score, i in bm25_search(query)[:5]]    # detective's shortlist
    vec_ids  = [i for score, i in vector_search(query)[:5]]  # map guide's shortlist
    return rrf_fuse([bm25_ids, vec_ids], top=top), bm25_ids, vec_ids

for q in ["E-4042", "puppy playing outside"]:
    fused, bm25_ids, vec_ids = hybrid_search(q)
    print(f"\nQuery: {q!r}")
    print("  BM25 ranked ids  :", bm25_ids)
    print("  Vector ranked ids:", vec_ids)
    print("  -- fused --")
    for i, score in fused:
        print(f"  {score:.5f}  doc {i}: {corpus[i]}")

Both failures fixed: `E-4042` won by two-hands-beat-one (0.03252 > 0.01639), and with
BM25's list empty the `puppy` fusion degraded gracefully into pure vector order.

## PRODUCTION — Qdrant hybrid in one query

One collection, both vector types; each query = two prefetches (ask both experts) + RRF
fusion, all server-side. Qdrant's RRF uses a smaller k than 60, so absolute scores differ —
the ORDER is what rank fusion promises, and it matches.

Bonus: production BM25 stems `playing` → `play`, so it actually matches doc 2 here —
a softer detective than our from-scratch one.

In [ ]:
from qdrant_client import QdrantClient, models

DENSE  = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE = "Qdrant/bm25"

client = QdrantClient(":memory:")                   # real server: QdrantClient(url=...)

client.create_collection(
    collection_name="hybrid_demo",
    vectors_config={"dense": models.VectorParams(size=384, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)

client.upsert(
    collection_name="hybrid_demo",
    points=[
        models.PointStruct(
            id=i,
            vector={
                "dense":  models.Document(text=t, model=DENSE),
                "sparse": models.Document(text=t, model=SPARSE),
            },
            payload={"text": t},
        )
        for i, t in enumerate(corpus)
    ],
)

for q in ["E-4042", "puppy playing outside"]:
    hits = client.query_points(
        collection_name="hybrid_demo",
        prefetch=[
            models.Prefetch(query=models.Document(text=q, model=DENSE),  using="dense",  limit=5),
            models.Prefetch(query=models.Document(text=q, model=SPARSE), using="sparse", limit=5),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    print(f"\nQuery: {q!r}")
    for h in hits.points:
        print(f"  {h.score:.4f}  {h.payload['text']}")

## Is hybrid always better? Measure it.

Hybrid's promise is robustness across a MIXED query stream. On a corpus living entirely on
one expert's turf, the specialist alone can beat the committee (fusion averages opinions —
averaging in a worse one costs). Measure recall and precision per method on your own data.

**Next: reranking** — hybrid retrieves a good pool; a cross-encoder re-sorts it.